In [1]:
import numpy as np
from scipy.special import erf
from math import sqrt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler

# ============================================================
# WEEK 10 — FUNCTION 5 (TRANSPARENT / INTERPRETABLE v4)
# Goal: push toward local maxima while making decisions auditable:
#  - explicit data cleaning + reporting (clip to [0,1], duplicates)
#  - faster, reproducible candidate generation + dedup
#  - log-space surrogate + EI + small UCB hedge
#  - interpretability: local feature sensitivity at current best
# Output: x_next printed in 6 decimals (<= 6 decimals)
# ============================================================

# ============================================================
# 1) Data (as provided)
# ============================================================

X_raw = np.array([
    [0.19144708, 0.03819337, 0.60741781, 0.41458414],
    [0.75865295, 0.53651774, 0.65600038, 0.36034155],
    [0.43834987, 0.8043397 , 0.21024527, 0.15129482],
    [0.70605083, 0.53419196, 0.26424335, 0.48208755],
    [0.83647799, 0.19360965, 0.6638927 , 0.78564888],
    [0.68343225, 0.11866264, 0.82904591, 0.56757661],
    [0.55362148, 0.66734998, 0.32380582, 0.81486975],
    [0.35235627, 0.32224153, 0.11697937, 0.47311252],
    [0.15378571, 0.72938169, 0.42259844, 0.44307417],
    [0.46344227, 0.63002451, 0.10790646, 0.9576439 ],
    [0.67749115, 0.35850951, 0.47959222, 0.07288048],
    [0.58397341, 0.14724265, 0.34809746, 0.42861465],
    [0.30688872, 0.31687813, 0.62263448, 0.09539906],
    [0.51114177, 0.817957  , 0.72871042, 0.11235362],
    [0.43893338, 0.77409176, 0.37816709, 0.93369621],
    [0.22418902, 0.84648049, 0.87948418, 0.87851568],
    [0.72526172, 0.47987049, 0.08894684, 0.75976022],
    [0.35548161, 0.63961937, 0.41761768, 0.12260384],
    [0.11987923, 0.86254031, 0.64333133, 0.84980383],
    [0.12688467, 0.15342962, 0.77016219, 0.19051811],
    [0.936477  , 0.96254   , 0.979484  , 1.057643  ],
    [9.99999e-01, 9.99999e-01, 1.00000e-06, 1.00000e-06],
    [0.969909, 0.832442, 0.21234 , 0.181826],
    [0.941016, 0.898976, 0.944821, 0.820504],
    [0.797068, 0.977185, 0.882682, 0.998829],
    [0.967723, 0.993021, 0.944062, 0.958127],
    [0.967724, 0.993022, 0.944062, 0.958128],
    [0.999987, 1.000000, 1.000000, 1.000000],
    [1.000000, 1.000000, 0.989543, 1.000000]
], dtype=float)

y_raw = np.array([
    6.44434399e+01, 1.83013796e+01, 1.12939795e-01, 4.21089813e+00,
    2.58370525e+02, 7.84343889e+01, 5.75715369e+01, 1.09571876e+02,
    8.84799176e+00, 2.33223610e+02, 2.44230883e+01, 6.44201468e+01,
    6.34767158e+01, 7.97291299e+01, 3.55806818e+02, 1.08885962e+03,
    2.88667516e+01, 4.51815703e+01, 4.31612757e+02, 9.97233189e+00,
    7713.373609304799, 1616.6257474282386, 563.3093235824822, 3461.904255719222,
    4201.359836533477, 6351.888673363732, 6351.934585346949, 8662.230634792088,
    8464.140059698084
], dtype=float).reshape(-1, 1)

# ============================================================
# 2) Utilities: normal CDF/PDF, formatting, data audits
# ============================================================

def normal_cdf(x):
    return 0.5 * (1.0 + erf(x / sqrt(2.0)))

def normal_pdf(x):
    return np.exp(-0.5 * x**2) / np.sqrt(2.0 * np.pi)

def format_vec_6(x):
    return "[" + ", ".join(f"{v:.6f}" for v in np.asarray(x).ravel()) + "]"

def audit_and_clean_X(X, clip_low=0.0, clip_high=1.0, round_dup=9):
    """
    Transparency:
    - Clip inputs into [0,1] bounds (required by problem statement).
    - Report how many values were out-of-bounds.
    - Report approximate duplicates (rounded to round_dup decimals).
    """
    X = np.asarray(X, dtype=float)
    oob = np.sum((X < clip_low) | (X > clip_high))
    Xc = np.clip(X, clip_low, clip_high)

    # Approx duplicate count using rounding
    Xr = np.round(Xc, round_dup)
    _, counts = np.unique(Xr, axis=0, return_counts=True)
    dup_groups = int(np.sum(counts > 1))
    dup_points = int(np.sum(counts[counts > 1] - 1))  # number of "extra" duplicates

    report = {
        "oob_values_clipped": int(oob),
        "dup_groups_approx": dup_groups,
        "dup_points_approx": dup_points,
        "round_dup_decimals": round_dup,
    }
    return Xc, report

# ============================================================
# 3) Surrogate: small MLP + dropout + early stopping + ensemble
# ============================================================

class MLPRegressor(nn.Module):
    def __init__(self, input_dim, hidden=(64, 32), dropout_p=0.12):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout_p)]
            prev = h
        layers += [nn.Linear(prev, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

def train_model(
    model, X_train, y_train,
    epochs=900, batch_size=8, lr=2e-3, weight_decay=1e-4,
    patience=140, min_delta=1e-6, seed=0
):
    torch.manual_seed(seed)
    np.random.seed(seed)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    n = X_train.shape[0]
    idx = np.random.permutation(n)
    n_val = max(4, int(0.2 * n))
    val_idx, tr_idx = idx[:n_val], idx[n_val:]

    Xtr, ytr = X_train[tr_idx], y_train[tr_idx]
    Xva, yva = X_train[val_idx], y_train[val_idx]

    tr_ds = TensorDataset(torch.tensor(Xtr, dtype=torch.float32),
                          torch.tensor(ytr, dtype=torch.float32))
    tr_loader = DataLoader(tr_ds, batch_size=batch_size, shuffle=True)

    Xva_t = torch.tensor(Xva, dtype=torch.float32).to(device)
    yva_t = torch.tensor(yva, dtype=torch.float32).to(device)

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()

    best_val = float("inf")
    best_state = None
    bad = 0

    for _ in range(epochs):
        model.train()
        for xb, yb in tr_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()

        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(Xva_t), yva_t).item()

        if val_loss < best_val - min_delta:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, best_val

def mc_dropout_predict(model, X, n_samples=40):
    device = next(model.parameters()).device
    model.train()  # dropout ON
    X_t = torch.tensor(X, dtype=torch.float32).to(device)

    preds = []
    with torch.no_grad():
        for _ in range(n_samples):
            preds.append(model(X_t).cpu().numpy().reshape(-1))
    preds = np.vstack(preds)

    mu = preds.mean(axis=0)
    std = preds.std(axis=0) + 1e-9
    return mu, std

def ensemble_predict(models, X_scaled, n_mc=40):
    mus, sigs = [], []
    for m in models:
        mu, sd = mc_dropout_predict(m, X_scaled, n_samples=n_mc)
        mus.append(mu)
        sigs.append(sd)

    mus = np.vstack(mus)
    sigs = np.vstack(sigs)

    mean = mus.mean(axis=0)
    var = (sigs**2).mean(axis=0) + mus.var(axis=0)
    std = np.sqrt(np.maximum(var, 1e-12))
    return mean, std

# ============================================================
# 4) Acquisition in LOG SPACE: log-EI + small log-UCB hedge
# ============================================================

def expected_improvement(mu, sigma, best_mu, xi=0.001):
    mu = np.asarray(mu)
    sigma = np.asarray(sigma)

    improvement = mu - best_mu - xi
    z = improvement / sigma

    ei = np.zeros_like(mu)
    mask = sigma > 1e-9
    ei[mask] = improvement[mask] * normal_cdf(z[mask]) + sigma[mask] * normal_pdf(z[mask])
    ei = np.maximum(ei, 0.0)
    ei[~mask] = 0.0
    return ei

# ============================================================
# 5) Candidate generation + FAST dedup (transparent + reproducible)
# ============================================================

def sobol_candidates(n, dim, seed=0):
    engine = torch.quasirandom.SobolEngine(dimension=dim, scramble=True, seed=seed)
    return engine.draw(n).cpu().numpy()

def fast_dedup_candidates(C, X_existing, round_decimals=8):
    """
    Fast dedup:
    - Round candidates + existing points
    - Drop any candidate matching an existing point
    - Drop duplicates among candidates via np.unique
    """
    C = np.clip(C, 0.0, 1.0)
    C_round = np.round(C, round_decimals)
    X_round = np.round(np.clip(X_existing, 0.0, 1.0), round_decimals)

    existing_set = set(map(tuple, X_round))
    mask = np.array([tuple(row) not in existing_set for row in C_round], dtype=bool)
    C_round = C_round[mask]

    # Unique within candidates
    if C_round.size == 0:
        return C_round
    _, idx = np.unique(C_round, axis=0, return_index=True)
    return C_round[np.sort(idx)]

def propose_next_point(
    models, x_scaler, y_scaler,
    X_raw_clean, y_raw,
    seed=123,
    xi=0.001,
    topk_audit=8
):
    dim = X_raw_clean.shape[1]
    rng = np.random.default_rng(seed)
    n_obs = X_raw_clean.shape[0]

    best_idx = int(np.argmax(y_raw))
    best_x = X_raw_clean[best_idx]
    best_y = float(y_raw[best_idx])
    best_log = float(np.log1p(best_y))

    # Week 10 compute trade-off (transparent):
    # With ~20-30 points, extra compute often gives diminishing returns.
    # Keep candidate budgets moderate but still corner-biased.
    n_global = 1800
    n_local  = 1400
    n_beta   = 900
    n_corner = 700

    # Shrinking trust region (more exploitation as n grows)
    local_radius = float(np.clip(0.09 / np.sqrt(max(1.0, n_obs / 10.0)), 0.02, 0.07))

    Cg = sobol_candidates(n_global, dim, seed=seed)
    Cl = np.clip(best_x + rng.normal(0.0, local_radius, size=(n_local, dim)), 0.0, 1.0)
    Cb = rng.beta(9.0, 1.0, size=(n_beta, dim))  # stronger bias to 1.0 than Week 9
    Cc = np.clip(1.0 - np.abs(rng.normal(0.0, 0.015, size=(n_corner, dim))), 0.0, 1.0)

    C = np.vstack([Cg, Cl, Cb, Cc])
    C = fast_dedup_candidates(C, X_raw_clean, round_decimals=8)

    if C.shape[0] == 0:
        # Fallback: tiny jitter around best_x
        C = np.clip(best_x + rng.normal(0.0, 1e-3, size=(10, dim)), 0.0, 1.0)
        C = fast_dedup_candidates(C, X_raw_clean, round_decimals=8)

    # Predict in scaled X
    C_scaled = x_scaler.transform(C)

    # Model predicts in standardised log1p(y) space
    mu_s, std_s = ensemble_predict(models, C_scaled, n_mc=40)

    # Unscale to log1p(y)
    mu_log = y_scaler.inverse_transform(mu_s.reshape(-1, 1)).reshape(-1)
    std_log = std_s * y_scaler.scale_[0]

    # log-EI
    ei_log = expected_improvement(mu_log, std_log, best_log, xi=xi)

    # log-UCB hedge (reduced as n grows)
    beta = float(np.clip(0.8 / np.sqrt(max(1.0, n_obs)), 0.08, 0.25))
    ucb_log = mu_log + beta * std_log

    # Combine (transparent normalization)
    ei_n = ei_log / (np.max(ei_log) + 1e-12)
    ucb_n = (ucb_log - np.min(ucb_log)) / (np.max(ucb_log) - np.min(ucb_log) + 1e-12)
    score = ei_n + 0.20 * ucb_n

    idx = int(np.argmax(score))

    # Audit: show top-k candidates
    top_idx = np.argsort(score)[::-1][:topk_audit]
    audit_rows = []
    for j in top_idx:
        audit_rows.append({
            "x": C[j],
            "ei_log": float(ei_log[j]),
            "ucb_log": float(ucb_log[j]),
            "score": float(score[j]),
        })

    return {
        "x_best": best_x,
        "y_best": best_y,
        "x_next": C[idx],
        "beta_ucb": beta,
        "xi": float(xi),
        "local_radius": local_radius,
        "n_candidates": int(C.shape[0]),
        "mix": {"global": n_global, "local": n_local, "beta": n_beta, "corner": n_corner},
        "audit_top": audit_rows
    }

# ============================================================
# 6) Interpretability: local gradient sensitivity at x_best
# ============================================================

def local_sensitivity(models, x_scaler, y_scaler, x_point_raw):
    """
    Reports relative feature influence around x_point_raw by computing
    gradient of predicted log1p(y) w.r.t. scaled inputs, then mapping back
    (approx) via scaler scale.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    x_scaled = x_scaler.transform(np.asarray(x_point_raw, dtype=float).reshape(1, -1))
    x_t = torch.tensor(x_scaled, dtype=torch.float32, requires_grad=True).to(device)

    grads = []
    for m in models:
        m = m.to(device)
        m.eval()
        y_s = m(x_t)  # standardized log space
        # convert to log space (inverse standard scaler):
        # log = y_s * scale + mean
        log_pred = y_s * float(y_scaler.scale_[0]) + float(y_scaler.mean_[0])
        g = torch.autograd.grad(log_pred.sum(), x_t, retain_graph=False)[0]
        grads.append(g.detach().cpu().numpy().reshape(-1))

    g_mean = np.mean(np.vstack(grads), axis=0)
    # account for x scaling: raw approx gradient proportional to g_mean / x_scaler.scale_
    raw_grad = g_mean / (x_scaler.scale_ + 1e-12)
    rel = np.abs(raw_grad)
    rel = rel / (rel.sum() + 1e-12)

    return {
        "mean_grad_scaled": g_mean,
        "relative_feature_influence": rel
    }

# ============================================================
# 7) Main
# ============================================================

def main():
    # --- Transparency: clean X to [0,1] and report
    X_clean, report = audit_and_clean_X(X_raw, clip_low=0.0, clip_high=1.0, round_dup=9)

    # Stabiliser: model log1p(y)
    y_log = np.log1p(y_raw)

    x_scaler = StandardScaler()
    y_scaler = StandardScaler()

    X = x_scaler.fit_transform(X_clean)
    y = y_scaler.fit_transform(y_log)

    # Week 10 compute/robustness: modest ensemble
    # (enough to reduce variance, not so large it hides logic)
    seeds = [0, 1, 2]
    models = []
    val_losses = []
    for s in seeds:
        m = MLPRegressor(input_dim=4, hidden=(64, 32), dropout_p=0.12)
        m, v = train_model(
            m, X, y,
            epochs=900,
            batch_size=8,
            lr=2e-3,
            weight_decay=1e-4,
            patience=140,
            min_delta=1e-6,
            seed=s
        )
        models.append(m)
        val_losses.append(v)

    r = propose_next_point(
        models, x_scaler, y_scaler,
        X_clean, y_raw,
        seed=123,
        xi=0.001,
        topk_audit=8
    )

    sens = local_sensitivity(models, x_scaler, y_scaler, r["x_best"])

    print("\n================ WEEK 10 FUNCTION 5 (TRANSPARENT v4) ================")

    print("\nDATA AUDIT / CLEANING")
    print("- out-of-bound values clipped to [0,1]:", report["oob_values_clipped"])
    print("- approx duplicate groups (rounded):", report["dup_groups_approx"])
    print("- approx duplicate extra points:", report["dup_points_approx"])
    print("- duplicate rounding decimals:", report["round_dup_decimals"])

    print("\nMODEL TRAINING (log1p(y) standardized)")
    print("Ensemble seeds:", seeds)
    print("Val losses (scaled log-space):", [float(f"{v:.6f}") for v in val_losses])

    print("\nCURRENT BEST OBSERVED")
    print("x_best =", format_vec_6(r["x_best"]), ", y_best =", f"{r['y_best']:.6f}")

    print("\nCANDIDATE MIX (TRANSPARENT)")
    print("- mix sizes:", r["mix"])
    print(f"- candidates used (after fast dedup) = {r['n_candidates']}")
    print(f"- local_radius = {r['local_radius']:.6f}")
    print(f"- xi (log-EI)  = {r['xi']:.6f}")
    print(f"- beta (UCB)   = {r['beta_ucb']:.6f}")

    print("\nINTERPRETABILITY (LOCAL SENSITIVITY AT x_best)")
    rel = sens["relative_feature_influence"]
    print("- relative feature influence (x1..x4):",
          "[" + ", ".join(f"{v:.3f}" for v in rel) + "]")

    print("\nAUDIT: TOP SCORING CANDIDATES (showing EI, UCB, score)")
    for k, row in enumerate(r["audit_top"], start=1):
        print(f"{k:02d}. x={format_vec_6(row['x'])}  "
              f"ei_log={row['ei_log']:.6f}  ucb_log={row['ucb_log']:.6f}  score={row['score']:.6f}")

    print("\nNEXT DATA POINT (6 decimals)")
    print("x_next =", format_vec_6(r["x_next"]))

if __name__ == "__main__":
    main()


C:\Users\veeja\AppData\Local\Temp\ipykernel_71552\3992614927.py:272: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  best_y = float(y_raw[best_idx])



================ WEEK 10 FUNCTION 5 (TRANSPARENT v4) ================

DATA AUDIT / CLEANING
- out-of-bound values clipped to [0,1]: 1
- approx duplicate groups (rounded): 0
- approx duplicate extra points: 0
- duplicate rounding decimals: 9

MODEL TRAINING (log1p(y) standardized)
Ensemble seeds: [0, 1, 2]
Val losses (scaled log-space): [0.335783, 0.03306, 0.083978]

CURRENT BEST OBSERVED
x_best = [0.999987, 1.000000, 1.000000, 1.000000] , y_best = 8662.230635

CANDIDATE MIX (TRANSPARENT)
- mix sizes: {'global': 1800, 'local': 1400, 'beta': 900, 'corner': 700}
- candidates used (after fast dedup) = 4719
- local_radius = 0.052850
- xi (log-EI)  = 0.001000
- beta (UCB)   = 0.148556

INTERPRETABILITY (LOCAL SENSITIVITY AT x_best)
- relative feature influence (x1..x4): [0.187, 0.282, 0.244, 0.286]

AUDIT: TOP SCORING CANDIDATES (showing EI, UCB, score)
01. x=[0.983627, 1.000000, 1.000000, 1.000000]  ei_log=0.319418  ucb_log=9.315753  score=1.199597
02. x=[1.000000, 1.000000, 1.000000, 0.9